# Khảo sát schema và quan hệ dữ liệu Airbnb Singapore – Mục 2.1
Notebook đọc dữ liệu gốc trong `data/raw`, không ghi đè hay chỉnh sửa dữ liệu nguồn.

Thư mục project khi tạo notebook: `E:/NKDL/Singapore_Airbnb`


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r'''E:\NKDL\Singapore_Airbnb''')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
FILES = {'Listings':'listings.csv.gz', 'Calendar':'calendar.csv.gz', 'Reviews':'reviews.csv.gz', 'Neighbourhoods':'neighbourhoods.csv'}
def read_source(name):
    p = RAW_DIR / FILES[name]
    return pd.read_csv(p, compression='gzip' if p.suffix == '.gz' else None, encoding='utf-8-sig', low_memory=False)
data = {name: read_source(name) for name in FILES}
{name: df.shape for name, df in data.items()}

## 1. Kiểm tra từng nguồn: cột, dtype ban đầu, kích thước, mẫu và phạm vi ngày


In [ ]:
date_cols = {'Listings':['last_scraped','calendar_last_scraped','first_review','last_review'], 'Calendar':['date'], 'Reviews':['date'], 'Neighbourhoods':[]}
schema_rows = []
for name, df in data.items():
    print(f'\n### {name}: {df.shape[0]:,} hàng x {df.shape[1]} cột')
    for col in df.columns:
        vals = df[col].dropna().astype(str).drop_duplicates().head(3).tolist()
        dmin = dmax = ''
        if col in date_cols[name]:
            d = pd.to_datetime(df[col], errors='coerce')
            if d.notna().any(): dmin, dmax = d.min().date().isoformat(), d.max().date().isoformat()
        schema_rows.append({'source':name,'column':col,'dtype_initial':str(df[col].dtype),'rows':len(df),'columns':len(df.columns),'null_count':int(df[col].isna().sum()),'unique_non_null':int(df[col].nunique()),'sample_values':' | '.join(vals),'date_min':dmin,'date_max':dmax})
    display(df.head(3))
schema_summary = pd.DataFrame(schema_rows)
display(schema_summary)

## 2. Grain và kiểm tra khóa/liên kết


In [ ]:
listings, calendar, reviews, neighbourhoods = (data[k] for k in ['Listings','Calendar','Reviews','Neighbourhoods'])
print('Listings grain: listing tại snapshot; id duy nhất:', listings['id'].is_unique if 'id' in listings else 'thiếu cột')
print('Calendar grain: listing-ngày; duplicate (listing_id,date):', calendar.duplicated(['listing_id','date']).sum() if {'listing_id','date'} <= set(calendar.columns) else 'thiếu cột')
print('Reviews grain: một review; duplicate id:', reviews['id'].duplicated().sum() if 'id' in reviews else 'thiếu cột')
print('Neighbourhoods grain: một khu vực; duplicate neighbourhood:', neighbourhoods['neighbourhood'].duplicated().sum() if 'neighbourhood' in neighbourhoods else 'thiếu cột')
listing_ids = set(listings['id'].dropna().astype(str))
for name, df in [('Calendar',calendar),('Reviews',reviews)]:
    ids = set(df['listing_id'].dropna().astype(str))
    print(name, 'listing_id không khớp Listings:', len(ids-listing_ids))
if 'neighbourhood_cleansed' in listings and 'neighbourhood' in neighbourhoods:
    a=set(listings['neighbourhood_cleansed'].dropna().astype(str)); b=set(neighbourhoods['neighbourhood'].dropna().astype(str))
    print('Neighbourhood chưa map:', sorted(a-b))
    print('Neighbourhood trong danh mục không xuất hiện ở Listings:', sorted(b-a))
print('Listings.neighbourhood NULL:', listings['neighbourhood'].isna().sum() if 'neighbourhood' in listings else 'thiếu cột')

## 3. Sơ đồ quan hệ sơ bộ
```mermaid
erDiagram
  NEIGHBOURHOODS ||--o{ LISTINGS : neighbourhood_cleansed
  LISTINGS ||--o{ CALENDAR : id_listing_id
  LISTINGS ||--o{ REVIEWS : id_listing_id
  NEIGHBOURHOODS { string neighbourhood PK }
  LISTINGS { int id PK
 string neighbourhood_cleansed FK }
  CALENDAR { int listing_id FK
 date date }
  REVIEWS { int id_review
 int listing_id FK }
```
**Lưu ý:** `Reviews.id` cần kiểm tra tính duy nhất trước khi xem là khóa chính; `Listings.neighbourhood` và `Listings.neighbourhood_cleansed` không được mặc định là cùng một trường.
